# MCP-Shield: ML Model Training, Statistical Diagnostics & Free Cloud Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahulxcodex/mcp-shield/blob/main/notebooks/mcp_shield_ml_training.ipynb)

This notebook trains and validates **Model A (Tabular Tool/Action Risk Model)** for **MCP-Shield v2.0** per the architecture specification in `ML_ROADMAP_GOOGLE_COLAB.md`.

### Pipeline Overview:
1. **Feature Engineering**: 42 versioned features across tool identity, capabilities, request entropy, behavioral transitions, and provenance.
2. **Rigorous Splits**: Train (70%), Validation (15%), and Holdout Test (15%) with unseen server and attack holdouts.
3. **Statistical Diagnostics**:
   - **Multicollinearity**: Variance Inflation Factor (VIF) & Pearson correlation.
   - **Homoscedasticity**: Breusch-Pagan Lagrange Multiplier test on residual error variance.
4. **Model Selection**: Logistic Regression vs. Random Forest vs. LightGBM vs. XGBoost.
5. **Overfitting Safeguards**: 5-Fold Stratified Cross-Validation and Regularization.
6. **Calibration**: Isotonic Probability Calibration optimizing Brier score and confidence reliability.
7. **Export & Serving**: Serializing to Joblib for 1-click deployment on Hugging Face Spaces (Free Tier) or Render.

## Step 1: Install Dependencies

In [ ]:
!pip install -q scikit-learn statsmodels xgboost lightgbm scipy joblib matplotlib seaborn

## Step 2: Import Libraries & Feature Schema Definition

In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc, brier_score_loss,
    classification_report, confusion_matrix, roc_curve
)

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

import xgboost as xgb
import lightgbm as lgb
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)

# 42 Versioned Features matching MCP-Shield FeatureExtractor
FEATURE_NAMES = [
    'tool_schema_complexity', 'tool_param_count', 'tool_cap_fs_read', 'tool_cap_fs_write',
    'tool_cap_process_spawn', 'tool_cap_network_egress', 'tool_cap_secret_access', 'tool_cap_db_access',
    'tool_destructive_capability', 'tool_capability_mismatch', 'tool_schema_drift', 'tool_publisher_trust',
    'tool_server_age_days', 'tool_historical_incidents',
    'req_payload_size_bytes', 'req_entropy', 'req_encoding_count', 'req_url_count',
    'req_ip_literals', 'req_special_ip_rep', 'req_shell_metachars', 'req_interpreter_transitions',
    'req_path_traversal_indicators', 'req_secret_findings', 'req_prompt_injection_signals',
    'seq_unique_tools_last_5', 'seq_unique_tools_last_10', 'seq_trans_read_to_network',
    'seq_trans_read_encode_network', 'seq_trans_db_to_export', 'seq_trans_db_export_upload',
    'seq_trans_fs_archive_upload', 'seq_trans_new_cap_external_dest', 'seq_velocity_ops_per_min',
    'seq_unseen_tool_transition',
    'prov_binary_hash_changed', 'prov_dep_graph_changed', 'prov_schema_fingerprint_changed',
    'prov_publisher_identity_score', 'prov_first_seen_days', 'prov_deployment_history_score',
    'prov_previous_violations'
]
print(f"Defined {len(FEATURE_NAMES)} MCP-Shield features.")

## Step 3: Dataset Generation & Real-World Attack Telemetry Synthesis

Simulates realistic MCP-Shield production traffic and known threat vectors (Prompt Injections, SSRF/Cloud Metadata, Command Injection, Path Traversal, Exfiltration, Tool Poisoning).

In [ ]:
def generate_telemetry_corpus(n_samples=10000, benign_ratio=0.65):
    n_benign = int(n_samples * benign_ratio)
    n_attack = n_samples - n_benign
    
    train_servers = [f"mcp-srv-{i:03d}" for i in range(1, 41)]
    heldout_servers = [f"mcp-unseen-{i:03d}" for i in range(1, 11)]
    
    rows = []
    for _ in range(n_benign):
        srv = np.random.choice(train_servers + heldout_servers)
        server_age = int(np.random.uniform(15, 750))
        pub_trust = float(np.clip(np.random.beta(8, 2), 0.5, 1.0))
        rows.append({
            'tool_schema_complexity': int(np.random.randint(1, 8)),
            'tool_param_count': int(np.random.randint(1, 6)),
            'tool_cap_fs_read': 1 if np.random.rand() < 0.3 else 0,
            'tool_cap_fs_write': 1 if np.random.rand() < 0.08 else 0,
            'tool_cap_process_spawn': 1 if np.random.rand() < 0.03 else 0,
            'tool_cap_network_egress': 1 if np.random.rand() < 0.25 else 0,
            'tool_cap_secret_access': 0,
            'tool_cap_db_access': 1 if np.random.rand() < 0.15 else 0,
            'tool_destructive_capability': 0,
            'tool_capability_mismatch': 0,
            'tool_schema_drift': 0,
            'tool_publisher_trust': pub_trust,
            'tool_server_age_days': server_age,
            'tool_historical_incidents': 0,
            'req_payload_size_bytes': int(np.random.lognormal(5.5, 1.2)),
            'req_entropy': float(np.clip(np.random.normal(4.1, 0.5), 1.5, 6.0)),
            'req_encoding_count': 0 if np.random.rand() < 0.95 else 1,
            'req_url_count': int(np.random.choice([0, 1, 2], p=[0.75, 0.2, 0.05])),
            'req_ip_literals': 0,
            'req_special_ip_rep': 0,
            'req_shell_metachars': 0 if np.random.rand() < 0.98 else 1,
            'req_interpreter_transitions': 0,
            'req_path_traversal_indicators': 0,
            'req_secret_findings': 0,
            'req_prompt_injection_signals': 0,
            'seq_unique_tools_last_5': int(np.random.randint(1, 4)),
            'seq_unique_tools_last_10': int(np.random.randint(2, 6)),
            'seq_trans_read_to_network': 1 if np.random.rand() < 0.05 else 0,
            'seq_trans_read_encode_network': 0,
            'seq_trans_db_to_export': 0,
            'seq_trans_db_export_upload': 0,
            'seq_trans_fs_archive_upload': 0,
            'seq_trans_new_cap_external_dest': 0,
            'seq_velocity_ops_per_min': float(np.random.exponential(12.0)),
            'seq_unseen_tool_transition': 0 if np.random.rand() < 0.96 else 1,
            'prov_binary_hash_changed': 0,
            'prov_dep_graph_changed': 0,
            'prov_schema_fingerprint_changed': 0,
            'prov_publisher_identity_score': pub_trust,
            'prov_first_seen_days': int(np.random.uniform(10, server_age)),
            'prov_deployment_history_score': float(np.clip(np.random.beta(7, 2), 0.4, 1.0)),
            'prov_previous_violations': 0,
            'server_id': srv,
            'family': 'BENIGN',
            'is_attack': 0
        })
        
    families = ['PROMPT_INJECTION', 'COMMAND_INJECTION', 'PATH_TRAVERSAL', 'SSRF_METADATA', 'CREDENTIAL_THEFT', 'DATA_EXFILTRATION', 'TOOL_POISONING']
    for _ in range(n_attack):
        fam = np.random.choice(families)
        srv = np.random.choice(train_servers + heldout_servers)
        item = rows[0].copy()
        item.update({
            'server_id': srv,
            'family': fam,
            'is_attack': 1,
            'tool_historical_incidents': int(np.random.choice([0, 1, 2], p=[0.6, 0.3, 0.1]))
        })
        if fam == 'PROMPT_INJECTION':
            item['req_prompt_injection_signals'] = int(np.random.randint(1, 5))
            item['req_entropy'] = float(np.random.uniform(5.2, 7.8))
            item['req_payload_size_bytes'] = int(np.random.lognormal(7.2, 0.8))
        elif fam == 'COMMAND_INJECTION':
            item['req_shell_metachars'] = int(np.random.randint(2, 8))
            item['req_interpreter_transitions'] = int(np.random.choice([1, 2], p=[0.7, 0.3]))
            item['tool_cap_process_spawn'] = 1
        elif fam == 'PATH_TRAVERSAL':
            item['req_path_traversal_indicators'] = int(np.random.randint(2, 6))
            item['tool_cap_fs_read'] = 1
        elif fam == 'SSRF_METADATA':
            item['req_special_ip_rep'] = 1
            item['tool_cap_network_egress'] = 1
            item['seq_trans_read_to_network'] = 1
        elif fam == 'CREDENTIAL_THEFT':
            item['req_secret_findings'] = int(np.random.randint(1, 5))
            item['tool_cap_secret_access'] = 1
        elif fam == 'DATA_EXFILTRATION':
            item['seq_trans_db_to_export'] = 1
            item['seq_trans_db_export_upload'] = 1
            item['tool_cap_network_egress'] = 1
            item['req_payload_size_bytes'] = int(np.random.lognormal(10.5, 1.0))
        elif fam == 'TOOL_POISONING':
            item['tool_schema_drift'] = 1
            item['prov_schema_fingerprint_changed'] = 1
            item['tool_publisher_trust'] = float(np.random.uniform(0.1, 0.4))
        rows.append(item)
        
    df = pd.DataFrame(rows).sample(frac=1.0, random_state=42).reset_index(drop=True)
    return df, heldout_servers

df, heldout_servers = generate_telemetry_corpus(10000)
print(f"Loaded dataset: {len(df)} rows across {df['family'].nunique()} families.")
display(df['family'].value_counts())

## Step 4: Train / Validation / Holdout Test Split

We isolate out-of-domain servers strictly into the test set to evaluate real-world generalization.

In [ ]:
is_test_server = df['server_id'].isin(heldout_servers[:5])
test_heldout = df[is_test_server]
remaining = df[~is_test_server]

train_df, val_df = train_test_split(remaining, test_size=0.20, random_state=42, stratify=remaining['is_attack'])
test_df = pd.concat([test_heldout, val_df.sample(frac=0.3, random_state=42)]).reset_index(drop=True)
val_df = val_df.drop(test_df.index, errors='ignore').reset_index(drop=True)

print(f"Training set:   {len(train_df)} samples ({train_df['is_attack'].mean()*100:.1f}% attacks)")
print(f"Validation set: {len(val_df)} samples ({val_df['is_attack'].mean()*100:.1f}% attacks)")
print(f"Holdout Test:   {len(test_df)} samples ({test_df['is_attack'].mean()*100:.1f}% attacks)")

## Step 5: Multicollinearity Diagnosis (Variance Inflation Factor - VIF)

Multicollinearity inflates standard errors and destabilizes feature attribution. A score of $\text{VIF} < 5.0$ indicates low collinearity, while $\text{VIF} \ge 10.0$ requires elimination.

In [ ]:
continuous_feats = [
    'tool_schema_complexity', 'tool_param_count', 'tool_publisher_trust',
    'tool_server_age_days', 'req_payload_size_bytes', 'req_entropy',
    'seq_unique_tools_last_5', 'seq_unique_tools_last_10',
    'seq_velocity_ops_per_min', 'prov_publisher_identity_score',
    'prov_first_seen_days', 'prov_deployment_history_score'
]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(train_df[continuous_feats]), columns=continuous_feats)
X_const = sm.add_constant(X_scaled)

vif_data = []
for i, col in enumerate(continuous_feats):
    vif = variance_inflation_factor(X_const.values, i + 1)
    vif_data.append({'Feature': col, 'VIF': round(float(vif), 3)})
    
vif_df = pd.DataFrame(vif_data).sort_values(by='VIF', ascending=False)
display(vif_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=vif_df, x='VIF', y='Feature', palette='viridis')
plt.axvline(5.0, color='orange', linestyle='--', label='Warning Threshold (5.0)')
plt.axvline(10.0, color='red', linestyle='--', label='Severe Multicollinearity (10.0)')
plt.title('Variance Inflation Factor (VIF) by Feature')
plt.legend()
plt.tight_layout()
plt.show()

## Step 6: Homoscedasticity Diagnosis (Breusch-Pagan Test)

Homoscedasticity assumes constant residual variance $\operatorname{Var}(\epsilon_i | X) = \sigma^2$. If $p < 0.05$, heteroscedasticity is present, proving that non-linear tree ensembles with non-parametric calibration are statistically necessary.

In [ ]:
X_tr_all = train_df[FEATURE_NAMES].values
y_tr_all = train_df['is_attack'].values

X_scaled_all = sm.add_constant(StandardScaler().fit_transform(X_tr_all))
ols = sm.OLS(y_tr_all, X_scaled_all).fit()
resid = ols.resid

lm_stat, p_val, f_stat, f_pval = het_breuschpagan(resid, X_scaled_all)
print(f"Breusch-Pagan LM Stat: {lm_stat:.4f}, p-value: {p_val:.4e}")
print(f"F-Statistic:           {f_stat:.4f}, F p-value: {f_pval:.4e}")

plt.figure(figsize=(9, 4))
plt.scatter(ols.fittedvalues, resid, alpha=0.3, s=15, color='darkblue')
plt.axhline(0, color='red', linestyle='--')
plt.title('Residuals vs Fitted Values (Diagnostics for Homoscedasticity)')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.tight_layout()
plt.show()

## Step 7: Model Selection Benchmark

Benchmarking Logistic Regression, Random Forest, LightGBM, and XGBoost across PR-AUC, ROC-AUC, Generalization Gap, and Brier Score.

In [ ]:
X_train = train_df[FEATURE_NAMES].values
y_train = train_df['is_attack'].values
X_val = val_df[FEATURE_NAMES].values
y_val = val_df['is_attack'].values
X_test = test_df[FEATURE_NAMES].values
y_test = test_df['is_attack'].values

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=120, max_depth=8, learning_rate=0.08, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.08, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
}

results = []
for name, clf in models.items():
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train) if name == 'Logistic Regression' else X_train
    X_te = scaler.transform(X_test) if name == 'Logistic Regression' else X_test
    
    clf.fit(X_tr, y_train)
    train_acc = clf.score(X_tr, y_train)
    test_acc = clf.score(X_te, y_test)
    probs = clf.predict_proba(X_te)[:, 1]
    
    roc = roc_auc_score(y_test, probs)
    prec, rec, _ = precision_recall_curve(y_test, probs)
    pr_auc = auc(rec, prec)
    brier = brier_score_loss(y_test, probs)
    
    results.append({
        'Model': name,
        'Train Acc': round(train_acc, 4),
        'Test Acc': round(test_acc, 4),
        'Gen Gap': round(train_acc - test_acc, 4),
        'ROC-AUC': round(roc, 4),
        'PR-AUC': round(pr_auc, 4),
        'Brier Score': round(brier, 4)
    })
    
bench_df = pd.DataFrame(results).sort_values(by='PR-AUC', ascending=False)
display(bench_df)

## Step 8: Overfitting / Underfitting Assessment & Probability Calibration

We verify that the generalization gap is minimal (no overfitting) and calibrate output probabilities using Isotonic Regression.

In [ ]:
best_clf = models['XGBoost']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_clf, X_train, y_train, cv=skf, scoring='roc_auc')
print(f"5-Fold CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Isotonic Calibration
calibrated_clf = CalibratedClassifierCV(estimator=best_clf, cv='prefit', method='isotonic')
calibrated_clf.fit(X_train, y_train)

raw_probs = best_clf.predict_proba(X_test)[:, 1]
cal_probs = calibrated_clf.predict_proba(X_test)[:, 1]

print(f"Raw Brier Score:        {brier_score_loss(y_test, raw_probs):.4f}")
print(f"Calibrated Brier Score: {brier_score_loss(y_test, cal_probs):.4f}")

# Plot Calibration Curves
prob_true_raw, prob_pred_raw = calibration_curve(y_test, raw_probs, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_test, cal_probs, n_bins=10)

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], 'k:', label='Perfect Calibration')
plt.plot(prob_pred_raw, prob_true_raw, 's-', label='XGBoost (Uncalibrated)')
plt.plot(prob_pred_cal, prob_true_cal, 'o-', label='XGBoost (Isotonic Calibrated)')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives (Observed Frequency)')
plt.title('Reliability Diagram / Calibration Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 9: Model Serialization & Free Hosting Microservice

Save model artifacts to Joblib for instant deployment to Hugging Face Spaces (Free CPU tier) or Render.

In [ ]:
os.makedirs('models/export', exist_ok=True)
pkg = {
    'model_name': 'XGBoost',
    'model': calibrated_clf,
    'feature_names': FEATURE_NAMES,
    'version': 'v2.0.0'
}
joblib.dump(pkg, 'models/export/mcp_shield_risk_model.joblib')
print("Model successfully packaged in 'models/export/mcp_shield_risk_model.joblib'!")
print("Ready for free deployment on Hugging Face Spaces or Render.")